# درس 3: احتمال شرطی، استقلال و Bayes

متن کامل درس: [https://quantacademy.ir/courses/quant-math/conditional-probability-bayes/](https://quantacademy.ir/courses/quant-math/conditional-probability-bayes/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. احتمال شرطی

## ۲. قضیه Bayes

In [ ]:
import numpy as np
import pandas as pd

p_real, p_pass_real, p_pass_fake = 0.05, 0.80, 0.20
p_pass = p_pass_real * p_real + p_pass_fake * (1 - p_real)
posterior = p_pass_real * p_real / p_pass
print(round(p_pass, 3), round(posterior, 4))

In [ ]:
rng = np.random.default_rng(5)
n = 1_000_000
real = rng.random(n) < p_real
passed = np.where(real, rng.random(n) < p_pass_real, rng.random(n) < p_pass_fake)
print(round(passed.mean(), 4), round(real[passed].mean(), 4))

## ۳. آزمون سخت‌گیرانه‌تر و تکرار

In [ ]:
for label, pr, pf in [("Sharpe>1 (weak)", 0.80, 0.20), ("strict", 0.60, 0.02), ("strict + replicate", 0.60 * 0.60, 0.02 * 0.02)]:
    post = pr * p_real / (pr * p_real + pf * (1 - p_real))
    print(f"{label:<20} {post:.3f}")

## ۴. جدول توأم روی داده بازار

In [ ]:
URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
r = close.pct_change().dropna()
df = pd.DataFrame({"up": r > 0, "prev_up": (r > 0).shift(1)}).dropna().astype(bool)
joint = pd.crosstab(df["prev_up"], df["up"], normalize=True)
print(joint.round(3))
print(joint.sum(axis=1).round(3).to_dict(), joint.sum(axis=0).round(3).to_dict())
cond = pd.crosstab(df["prev_up"], df["up"], normalize="index")
print(cond.round(3))

In [ ]:
p_up = df["up"].mean()
n_after_up, n_after_down = df["prev_up"].sum(), (~df["prev_up"]).sum()
diff = cond.loc[True, True] - cond.loc[False, True]
se = np.sqrt(p_up * (1 - p_up) * (1 / n_after_up + 1 / n_after_down))
print(round(p_up, 3), round(diff, 3), round(se, 3), round(diff / se, 2))

## ۵. استقلال جهت، وابستگی ریسک

In [ ]:
big = r.abs() > 0.05
d = pd.DataFrame({"big": big, "prev_big": big.shift(1)}).dropna().astype(bool)
pb = d["big"].mean()
pbb = d.loc[d["prev_big"], "big"].mean()
n_prev_big = int(d["prev_big"].sum())
se_big = np.sqrt(pb * (1 - pb) / n_prev_big)
print(round(pb, 3), round(pbb, 3), n_prev_big, round((pbb - pb) / se_big, 2))

## ۶. امید شرطی

In [ ]:
ce = pd.DataFrame({"r": r, "prev": r.shift(1)}).dropna()
print(f"{ce['r'].mean():.5f} {ce['r'].abs().mean():.4f}")
for lab, mask in [("prev <= -5%", ce["prev"] <= -0.05), ("prev >= +5%", ce["prev"] >= 0.05), ("|prev| < 1%", ce["prev"].abs() < 0.01)]:
    print(lab, int(mask.sum()), f"{ce.loc[mask, 'r'].mean():.5f}", f"{ce.loc[mask, 'r'].abs().mean():.4f}")